## Library check

In [ ]:
import numpy as np
import torch
import torchaudio
import cLogger

#log = logger.Log("output/logs")

print("Torch version:", torch.__version__)
print("Torchaudio version:", torchaudio.__version__)

if torch.cuda.is_available():
    print("CUDA available")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA unavailable")


## Load Dataset

In [ ]:
import audiotools
import torch
import torchaudio
import cTransforms
from torch.utils.data import DataLoader

#Dataset parameters:
loader_params = {
    "batch_size": 16,
    "shuffle": True,
    "collate_function": audiotools.Batching.spectrogram_dynamic,
    "target_transform": cTransforms.NormalizeMinus(1, 7)
    }

#Spectogram parameters
spectogram_transform = torchaudio.transforms.Spectrogram(
    n_fft=1024,
    hop_length=160,
    win_length=400,
    window_fn=torch.hamming_window,
    normalized=True,
    power=2
    )

melspectogram_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=512,
    win_length=400,
    hop_length=160,
    window_fn=torch.hamming_window,
    power=2,
    n_mels=64,
    normalized=True,
    pad_mode="constant",
    mel_scale="htk"
)

mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=16000,
    n_mfcc=20,
    melkwargs={
        #"sample_rate": 16000,
        "n_fft": 512,
        "win_length": 400,
        "hop_length": 160,
        "window_fn": torch.hamming_window,
        "power": 2,
        "n_mels": 64,
        "normalized": True,
        "pad_mode": "constant",
        "mel_scale": "htk"
    }
)

#Train partition
msp_vad_train = audiotools.AudioDatasetVAD(
    r"C:\Datasets\_compiled\msp-podcast-2\labels_train_VAD.csv", 
    r"C:\Datasets\_compiled\msp-podcast-2\Train",
    transform=mfcc_transform,
    target_transform=loader_params["target_transform"]
    )
train_dataloader = DataLoader(
    msp_vad_train,
    batch_size=loader_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"]
    )

#Development(validation) partition
msp_vad_dev = audiotools.AudioDatasetVAD(
    r"C:\Datasets\_compiled\msp-podcast-2\labels_development_VAD.csv",
    r"C:\Datasets\_compiled\msp-podcast-2\Development",
    transform=spectogram_transform,
    target_transform=loader_params["target_transform"]
    )
dev_dataloader = DataLoader(
    msp_vad_dev,
    batch_size=loader_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"]
    )

train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")

sample = train_features[0]
label = train_labels[0]

spectogram_2_db = torchaudio.transforms.AmplitudeToDB(
    stype='power',
    #top_db=80
    )
#audiotools.Plot.spectrogram(spectogram_2_db(sample), ylabel="Frequency Bin", xlabel="Frame", cmap="viridis")
audiotools.Plot.mfcc(sample)
print("MFCC: ", sample)
print(f"Label: {label}")